# Evaluate Model on Unseen Test Set (Future Data)

Manchester.Ai_IBM Z Datathon 2026 Project. Runs on IBM LinuxONE.

**Goal:** Test the pipeline on completely unseen future data (September 9–10) to check how well the model generalises to later days. 

This is the most critical evaluation for the Datathon. A strong result here suggests the model hasn't simply memorised the training days (overfitting).

In [1]:
import os, json
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import average_precision_score

SHARED = os.path.expanduser('~/shared')
MODELS = f'{SHARED}/models'

## 1. Load Data and Filter for Test Split
By isolating the `'test'` split, we guarantee that the model is being evaluated on days it has never trained on and never used for threshold tuning. This keeps the evaluation honest in time. It does not separate accounts: the same accounts can appear in training and test days.

In [2]:
# 1. Load the pre-processed features
print("Loading historical feature data...")
test_data = pd.read_pickle(f'{SHARED}/feat_test.pkl')

print(f"Isolated {len(test_data):,} transactions from the unseen test period (Days 9-10).")
    

Loading historical feature data...
Isolated 845,157 transactions from the unseen test period (Days 9-10).


## 2. Load the model and score every payment
We load the XGBoost model and feature list saved by notebook 03 and score every payment in this split.

In [3]:
# 2. Load the model saved by notebook 03
with open(f'{MODELS}/config.json', 'r') as f:
    config = json.load(f)
FEATURES = config['features']

booster = xgb.Booster()
booster.load_model(f'{MODELS}/mule_xgb.json')

# 3. Score every payment
print("Running inference...")
test_data['xgb_score'] = booster.predict(xgb.DMatrix(test_data[FEATURES]))

# Flag the riskiest 1% of payments in this split (same definition as notebook 03)
k = int(len(test_data) * 0.01)
top = np.argsort(-test_data['xgb_score'].values, kind='stable')[:k]
test_data['is_flagged'] = 0
test_data.iloc[top, test_data.columns.get_loc('is_flagged')] = 1

Running inference...


## 3. Calculate Real-World Detection Metrics
Comparing these metrics to the validation notebook reveals the model's true stability. A narrow gap between validation and test recall suggests the model is stable over time. These are results on synthetic data; real-world accuracy would need real bank data.

In [4]:
# 4. Calculate Real-World Metrics
y_true = test_data['y'].values
preds = test_data['xgb_score'].values
flagged = test_data['is_flagged'].values

total_laundering = int(y_true.sum())
caught_laundering = int((y_true * flagged).sum())

# Precision-Recall AUC (Standard metric for highly imbalanced data)
pr_auc = average_precision_score(y_true, preds)

# Recall at 1% Alert Rate
recall_at_1_percent = caught_laundering / total_laundering

print("\n=== True Test Set Evaluation (Unseen Data) ===")
print(f"Total Transactions Evaluated : {len(test_data):,}")
print(f"Actual Laundering Events   : {total_laundering:,}")
print(f"Laundering Events Caught   : {caught_laundering:,}")
print("-" * 45)
print(f"Recall @ 1% alert rate     : {recall_at_1_percent:.2%}")
print(f"PR-AUC Score               : {pr_auc:.4f}")




=== True Test Set Evaluation (Unseen Data) ===
Total Transactions Evaluated : 845,157
Actual Laundering Events   : 954
Laundering Events Caught   : 663
---------------------------------------------
Recall @ 1% alert rate     : 69.50%
PR-AUC Score               : 0.4301
